# Retrieval Fundamentals: Queries, Candidates, Scores, and Top-K

Retrieval is the part of a RAG system that looks through a collection of content and selects the pieces most likely to help answer a question. This notebook builds that idea from first principles using a tiny, inspectable corpus and a deliberately simple lexical scoring rule.

The goal is not to build a production search engine. It is to make each step visible: what is searchable, which items become candidates, how candidates are ranked, what a score does and does not mean, and how we judge whether the returned results were useful.

## Learning goals

By the end, you should be able to:

- Trace a query from a corpus through candidate selection, scoring, ranking, and top-K output.
- Explain the difference between a document and a chunk, and why retrieval usually works over chunks.
- Read a ranking score as a method-specific signal rather than a probability of correctness.
- Identify ties, lexical mismatch, poor chunk boundaries, and missing evidence as distinct retrieval problems.
- Compute precision@K and recall@K from explicit relevance judgments.

This lesson uses word overlap only. The next lesson can replace this scoring rule with dense, embedding-based retrieval while keeping the same retrieval workflow.

## 1. What is retrieval?

Suppose a user asks a question about a collection of documents. A retriever searches the collection and returns a short, ranked list of passages that may contain useful evidence. A generator may later use those passages to write an answer, but generation is a separate step.

Retrieval does **not** guarantee that the answer exists, that a returned passage is correct, or that a language model will use the passage faithfully. It only produces candidates according to a search method and its configuration.

| Term | Meaning in this notebook |
| --- | --- |
| Corpus | The collection of searchable content. |
| Document | A source item, such as a policy page or handbook. |
| Chunk | A smaller passage from a document, stored and searched as one item. |
| Query | The user's information need expressed as text. |
| Candidate | A chunk considered by the ranking stage. |
| Score | A method-specific value used to compare candidates for a query. |
| Ranking | Ordering candidates from the method's strongest match to weakest. |
| Top-K | The first K ranked candidates returned to the next system stage. |

A practical pipeline is: **query → candidate generation → scoring → ranking → top-K results → downstream answer construction**. Some systems combine candidate generation and scoring; separating the ideas helps us diagnose failures.

### Retrieval versus generation

- **Retrieval asks:** Which passages in this collection are likely to help with this query?
- **Generation asks:** Given the query and selected evidence, what response should be written?

A fluent answer is not evidence that retrieval worked. For this notebook, inspect the returned chunks directly. We will not call a language model or generate answers, so every result can be traced to the corpus and ranking rule.

## 2. A small corpus we can inspect

Each row represents one chunk. The `chunk_id` is stable, `source` points to the parent document, and `topic` is descriptive metadata. The text is intentionally short so we can inspect matches by eye. In a real system, chunks often come from longer documents and may contain much more context.

In [4]:
corpus = [
    {'chunk_id': 'C01', 'source': 'account_help', 'topic': 'password',
     'text': 'To reset your password, choose Forgot password on the sign-in page. We send a reset link to your registered email address.'},
    {'chunk_id': 'C02', 'source': 'account_help', 'topic': 'security',
     'text': 'For account security, turn on two-factor authentication and never share your verification code with anyone.'},
    {'chunk_id': 'C03', 'source': 'shopping_policy', 'topic': 'refunds',
     'text': 'Refunds for eligible orders are returned to the original payment method after the returned item is inspected.'},
    {'chunk_id': 'C04', 'source': 'shipping_guide', 'topic': 'delivery',
     'text': 'Standard shipping usually takes five to seven business days. Express shipping is available at checkout.'},
    {'chunk_id': 'C05', 'source': 'shopping_policy', 'topic': 'returns',
     'text': 'Unopened products can be returned within thirty days of delivery. Include the order number in your return request.'},
    {'chunk_id': 'C06', 'source': 'tenant_handbook', 'topic': 'lease',
     'text': 'The security deposit is due when the lease is signed. The deposit amount is listed in the signed lease agreement.'},
    {'chunk_id': 'C07', 'source': 'cycling_guide', 'topic': 'safety',
     'text': 'Choose a bicycle helmet that fits snugly and sits level on the head. Replace a helmet after a serious impact.'},
    {'chunk_id': 'C08', 'source': 'account_help', 'topic': 'support',
     'text': 'Contact support if you cannot access the email address linked to your account.'},
    {'chunk_id': 'C09', 'source': 'food_guide', 'topic': 'allergies',
     'text': 'Read the ingredient label carefully if you have a food allergy. Contact the manufacturer when allergen information is unclear.'},
    {'chunk_id': 'C10', 'source': 'account_help', 'topic': 'limitations',
     'text': 'This page does not explain how to reset a password or recover an account; contact the service administrator for those steps.'},
]

print(f'Number of chunks: {len(corpus)}')
for chunk in corpus[:3]:
    print(chunk['chunk_id'], '-', chunk['source'], '-', chunk['text'])

Number of chunks: 10
C01 - account_help - To reset your password, choose Forgot password on the sign-in page. We send a reset link to your registered email address.
C02 - account_help - For account security, turn on two-factor authentication and never share your verification code with anyone.
C03 - shopping_policy - Refunds for eligible orders are returned to the original payment method after the returned item is inspected.


The corpus contains a deliberate trap in `C10`: it repeats words about resetting a password and recovering an account, but it does not provide those instructions. A lexical scoring rule may rank it highly even though it is not a useful answer. Search scores are clues for inspection, not truth labels.

### Why chunks instead of whole documents?

A long document may discuss many unrelated subjects. Searching and returning the whole document can bury the evidence, waste the downstream context window, and make it harder to identify which part supports an answer. Chunking makes focused passages searchable.

Chunking also creates a trade-off. A chunk that is too small can omit a definition or the context needed to interpret a sentence. A chunk that is too large can mix topics and dilute the match. Chunk boundaries are part of retrieval quality, not just a storage detail.

## 3. Make a query and tokenize the text

For this baseline, tokenization means converting text to lowercase and extracting runs of letters or digits. We treat a query as a set of tokens, so repeated occurrences of one word do not count repeatedly. This is intentionally simple: it does not understand synonyms, word meaning, negation, phrases, or morphology.

In [5]:
import re

def tokenize(text):
    return set(re.findall(r'[a-z0-9]+', text.lower()))

query = 'How do I reset my password?'

print('Query:', query)
print('Query tokens:', sorted(tokenize(query)))

Query: How do I reset my password?
Query tokens: ['do', 'how', 'i', 'my', 'password', 'reset']


The token set is `do`, `how`, `i`, `my`, `password`, and `reset`. Common words such as `do` and `my` are not removed yet. A production lexical retriever may use term weighting, stop-word handling, field boosts, stemming, phrase matching, or another strategy. Later in this module, BM25 will provide a stronger sparse-retrieval baseline.

## 4. Score every chunk with a transparent baseline

Define the score as the number of **unique query tokens** also present in the chunk:

`score(query, chunk) = |tokens(query) ∩ tokens(chunk)|`

A score of 2 means two distinct query tokens were shared. It does **not** mean a 2% chance, a 2-out-of-10 confidence value, or that the chunk contains the answer. This toy score is useful because we can explain every point; it is not intended to compete with BM25 or embedding similarity.

In [6]:
def overlap_score(query_text, chunk_text):
    return len(tokenize(query_text) & tokenize(chunk_text))

def score_corpus(query_text, chunks):
    scored = []
    for chunk in chunks:
        scored.append({
            **chunk,
            'score': overlap_score(query_text, chunk['text']),
        })
    return scored

scored_chunks = score_corpus(query, corpus)
for item in scored_chunks:
    print(item['chunk_id'], 'score =', item['score'])

C01 score = 2
C02 score = 0
C03 score = 0
C04 score = 0
C05 score = 0
C06 score = 0
C07 score = 0
C08 score = 0
C09 score = 0
C10 score = 3


The score is computed for every chunk, including zero-score chunks. This notebook's baseline scans the entire corpus, so its candidate set is the whole corpus. Larger systems may use an index to generate a smaller candidate set first; if a useful chunk never becomes a candidate, a later ranking step cannot recover it.

### Inspect why a chunk received its score

A useful debugging habit is to inspect the actual matching terms, not just the numeric score. The helper below shows which unique query tokens overlap with each chunk.

In [7]:
def matching_terms(query_text, chunk_text):
    return sorted(tokenize(query_text) & tokenize(chunk_text))

for item in scored_chunks:
    matches = matching_terms(query, item['text'])
    if matches:
        print(item['chunk_id'], item['score'], matches, '|', item['text'])

C01 2 ['password', 'reset'] | To reset your password, choose Forgot password on the sign-in page. We send a reset link to your registered email address.
C10 3 ['how', 'password', 'reset'] | This page does not explain how to reset a password or recover an account; contact the service administrator for those steps.


Notice that `C10` shares the terms `password` and `reset` with the query. Its words are a strong lexical match, but the passage explicitly says it does not explain the steps. A relevance judgment must consider whether a passage helps answer the information need, not merely whether it contains query words.

## 5. Rank candidates and return top-K

Ranking sorts candidates by score from highest to lowest. We also need a tie rule. Here, equal scores are ordered by `chunk_id` ascending. A deterministic tie rule makes demonstrations, tests, and debugging repeatable; it does not claim that the earlier ID is more relevant.

Top-K means returning at most K results from that ordered list. K is a retrieval configuration, not a statement about how many chunks truly contain useful evidence.

In [8]:
def rank_chunks(query_text, chunks):
    scored = score_corpus(query_text, chunks)
    return sorted(scored, key=lambda item: (-item['score'], item['chunk_id']))

def retrieve(query_text, chunks, k=3):
    if k < 0:
        raise ValueError('k must be non-negative')
    return rank_chunks(query_text, chunks)[:k]

ranked = rank_chunks(query, corpus)
for rank, item in enumerate(ranked, start=1):
    print(f"{rank}. {item['chunk_id']} score={item['score']} | {item['text']}")

1. C10 score=3 | This page does not explain how to reset a password or recover an account; contact the service administrator for those steps.
2. C01 score=2 | To reset your password, choose Forgot password on the sign-in page. We send a reset link to your registered email address.
3. C02 score=0 | For account security, turn on two-factor authentication and never share your verification code with anyone.
4. C03 score=0 | Refunds for eligible orders are returned to the original payment method after the returned item is inspected.
5. C04 score=0 | Standard shipping usually takes five to seven business days. Express shipping is available at checkout.
6. C05 score=0 | Unopened products can be returned within thirty days of delivery. Include the order number in your return request.
7. C06 score=0 | The security deposit is due when the lease is signed. The deposit amount is listed in the signed lease agreement.
8. C07 score=0 | Choose a bicycle helmet that fits snugly and sits level on the he

For the password-reset query, `C01` should rank first because it contains both `reset` and `password`, and it gives the reset procedure. `C10` also has both terms but is a deliberately misleading match. The tie-breaking rule determines the order between those equal-scoring chunks.

In [9]:
top_1 = retrieve(query, corpus, k=1)
top_3 = retrieve(query, corpus, k=3)
top_6 = retrieve(query, corpus, k=6)

print('K=1:', [item['chunk_id'] for item in top_1])
print('K=3:', [item['chunk_id'] for item in top_3])
print('K=6:', [item['chunk_id'] for item in top_6])

assert len(top_1) == 1
assert len(top_3) == 3
assert len(top_6) == 6
assert top_1[0]['chunk_id'] == 'C01'

K=1: ['C10']
K=3: ['C10', 'C01', 'C02']
K=6: ['C10', 'C01', 'C02', 'C03', 'C04', 'C05']


AssertionError: 

Increasing K returns more candidates, which can improve the chance of including useful evidence. It can also add irrelevant passages, increase latency and cost downstream, and make evidence harder to use. A small K risks missing relevant material; a large K does not automatically improve answer quality.

This baseline always returns exactly `min(K, number_of_chunks)` items, even when all remaining scores are zero. Real systems may apply a score threshold or return fewer results. Thresholds are method-specific and need validation; a raw overlap score is not a universal threshold scale.

### Ties and repeatable ordering

The single-token query `password` produces equal scores for multiple chunks. The stable ID tie-breaker gives a repeatable order. Depending on the search engine, ties may instead preserve insertion order, use internal index order, or be resolved in another way.

In [ ]:
tie_query = 'password'
tie_results = retrieve(tie_query, corpus, k=len(corpus))
positive_ties = [item for item in tie_results if item['score'] == 1]

print([(item['chunk_id'], item['score']) for item in positive_ties])
assert [item['chunk_id'] for item in positive_ties] == sorted(
    item['chunk_id'] for item in positive_ties
)

## 6. Try several queries and inspect failure modes

The same corpus and scoring rule can behave very differently as query wording changes. For each query below, inspect the top results and ask whether the returned text would actually help a person answer it.

In [ ]:
queries = [
    'How do I reset my password?',
    'How can I regain access after forgetting my login secret?',
    'What is the return period for unopened products?',
    'When do I pay the security deposit for a lease?',
    'Where do I find bicycle head protection advice?',
]

for query_text in queries:
    results = retrieve(query_text, corpus, k=3)
    print(f'\nQuery: {query_text}')
    for item in results:
        print(f"  {item['chunk_id']} score={item['score']} | {item['text']}")

### What should you notice?

1. **Good lexical match:** The return-period query shares several terms with `C05`, which contains the return window.
2. **A different topic can still match:** The lease query should favor `C06` because its wording overlaps the question.
3. **Lexical mismatch:** The login-secret query expresses the password-reset need without using the corpus words `password` or `reset`. This simple baseline may fail to retrieve `C01` near the top even though it is the right evidence. Embeddings are designed to help with semantic similarity, although they also have failure modes.
4. **Vocabulary mismatch:** The bicycle query uses `head protection`, while the chunk says `helmet`. Matching meaning across different words is not something this overlap score can do.
5. **The score can be fooled:** A result can share many words but contradict the user's need, as `C10` demonstrates for password reset.

### Candidate generation is not ranking

In this notebook, every corpus chunk is scored. Imagine instead that a first-stage candidate generator returned only `C02`, `C03`, and `C10`. Even a perfect later ranker could not return `C01`, because it never received it as a candidate. This illustrates two different questions:

- **Candidate recall:** Did the search stage include the useful chunk at all?
- **Ranking quality:** Given the candidates, did the useful chunk appear near the top?

Use the following cell to simulate candidate omission. The candidate list is chosen for demonstration, not produced by a real index.

In [10]:
limited_candidates = [chunk for chunk in corpus if chunk['chunk_id'] in {'C02', 'C03', 'C10'}]
full_results = retrieve('How do I reset my password?', corpus, k=3)
limited_results = retrieve('How do I reset my password?', limited_candidates, k=3)

print('Full corpus:', [item['chunk_id'] for item in full_results])
print('Limited candidates:', [item['chunk_id'] for item in limited_results])
assert 'C01' in [item['chunk_id'] for item in full_results]
assert 'C01' not in [item['chunk_id'] for item in limited_results]

Full corpus: ['C10', 'C01', 'C02']
Limited candidates: ['C10', 'C02', 'C03']


### Chunk granularity can change the evidence

Suppose an answer is split across two adjacent chunks: one chunk names a security deposit, while the next says when it is due. Retrieving only one chunk may omit the relation. Conversely, joining every topic into one huge chunk can make unrelated query terms match and return excessive context.

Real chunking strategies try to preserve useful context, sometimes with overlap or parent-child links. The best size depends on the documents, retrieval method, and downstream context budget; there is no universally correct chunk size.

## 7. Relevance judgments: what counts as useful?

A search system cannot be evaluated from scores alone. We need example queries and human judgments about which chunks are relevant to each information need. For this small exercise, relevant means the chunk provides useful evidence to answer the query, not merely that it shares words.

The judgments below are an illustrative ground-truth set for the teaching queries. Real evaluation labels should be created carefully, ideally by more than one reviewer for ambiguous tasks.

In [11]:
relevance_judgments = {
    'How do I reset my password?': {'C01'},
    'What is the return period for unopened products?': {'C05'},
    'When do I pay the security deposit for a lease?': {'C06'},
}

for query_text, relevant_ids in relevance_judgments.items():
    print(query_text, '->', sorted(relevant_ids))

How do I reset my password? -> ['C01']
What is the return period for unopened products? -> ['C05']
When do I pay the security deposit for a lease? -> ['C06']


## 8. Evaluate a ranking with precision@K and recall@K

For one query and a chosen K:

- **Precision@K** = relevant chunks among the first K ÷ number of returned chunks. It answers: how much of the returned list was useful?
- **Recall@K** = relevant chunks among the first K ÷ all known relevant chunks for the query. It answers: how much of the known useful evidence did we find?

These definitions assume the relevance set is known and non-empty. Here we validate that assumption explicitly. When fewer than K items exist, the precision denominator is the number actually returned, not the requested K.

In [12]:
def precision_at_k(results, relevant_ids, k):
    if k <= 0:
        raise ValueError('k must be positive')
    if not results[:k]:
        return 0.0
    hits = sum(item['chunk_id'] in relevant_ids for item in results[:k])
    return hits / len(results[:k])

def recall_at_k(results, relevant_ids, k):
    if k <= 0:
        raise ValueError('k must be positive')
    if not relevant_ids:
        raise ValueError('relevant_ids must not be empty')
    hits = sum(item['chunk_id'] in relevant_ids for item in results[:k])
    return hits / len(relevant_ids)

for query_text, relevant_ids in relevance_judgments.items():
    results = rank_chunks(query_text, corpus)
    print(f'\n{query_text}')
    for k in (1, 3, 5):
        print(f'  P@{k}={precision_at_k(results, relevant_ids, k):.2f}, '
              f'R@{k}={recall_at_k(results, relevant_ids, k):.2f}')


How do I reset my password?
  P@1=0.00, R@1=0.00
  P@3=0.33, R@3=1.00
  P@5=0.20, R@5=1.00

What is the return period for unopened products?
  P@1=1.00, R@1=1.00
  P@3=0.33, R@3=1.00
  P@5=0.20, R@5=1.00

When do I pay the security deposit for a lease?
  P@1=1.00, R@1=1.00
  P@3=0.33, R@3=1.00
  P@5=0.20, R@5=1.00


For a query with one relevant chunk, recall@K becomes 1 as soon as that chunk appears in the top K. Precision may decrease as K grows if extra results are irrelevant. With multiple relevant chunks, recall can continue to rise as more are found.

These metrics are useful but incomplete. They depend on the quality and coverage of the labels, treat all relevant chunks equally, and do not measure whether the eventual answer is faithful. Later evaluation work can add ranking-sensitive metrics such as mean reciprocal rank or nDCG, along with end-to-end answer evaluation.

## 9. Score interpretation and system choices

A score is meaningful only in the context of the scoring method and configuration that produced it.

- This overlap score is a count of shared unique terms.
- BM25 produces a weighted lexical relevance score whose behavior depends on the corpus and implementation.
- Dense retrieval commonly uses a distance or similarity between query and chunk embeddings.
- Different methods may use opposite directions: for one metric, larger is better; for a distance, smaller is better.

Do not compare raw values from different methods as if they were calibrated probabilities. Even within one method, score distributions can shift with query wording, corpus content, embedding model, index settings, or filters. Thresholds should be evaluated on representative examples.

### Common retrieval failure checklist

| Failure | What it looks like | Possible next step |
| --- | --- | --- |
| Lexical mismatch | Useful passage uses different words or a synonym. | Try dense retrieval, query expansion, or hybrid retrieval. |
| Misleading overlap | A passage repeats query terms but does not answer it or contradicts it. | Improve ranking, inspect context, or rerank candidates. |
| Missing candidate | The useful chunk was excluded before ranking. | Improve candidate generation or broaden the search. |
| Poor chunk boundary | The retrieved passage lacks nearby context needed to interpret it. | Adjust chunking, overlap, or return parent context. |
| Inappropriate K | Useful chunks are cut off, or too many weak chunks are returned. | Tune K using labeled queries and downstream constraints. |
| Missing evidence | No chunk in the corpus contains the answer. | Report insufficient evidence rather than inventing one. |

No retrieval strategy can find evidence that is absent from the indexed collection. A reliable RAG system needs a way to recognize uncertainty and avoid presenting unsupported content as fact.

## 10. Exercises

Try each task before opening the optional solution cell below it.

1. Write a query that retrieves `C05` first. Which words made the match easy?
2. Write a paraphrase of the password-reset question that avoids the words `password` and `reset`. Does `C01` still rank first?
3. Find a query for which `C10` receives a high score despite not providing the requested instructions. Explain why the score is misleading.
4. Change K from 1 to 5 for the lease query. Which additional chunks enter the result list, and how do precision@K and recall@K change?
5. Add a new chunk that correctly answers the password-reset question using wording unlike `C01`. Does this baseline retrieve it for a paraphrase?
6. Propose one improvement to the retrieval system and one change to the chunking or corpus. State what failure each is meant to address.

### Optional exercise solution: inspect custom queries

The answers are not unique. Run this cell with your own query ideas and explain the ranking using the matching terms and passage meaning.

In [13]:
exercise_query = 'What is the return window for unopened items?'
for item in retrieve(exercise_query, corpus, k=4):
    print(item['chunk_id'], item['score'], matching_terms(exercise_query, item['text']))
    print(' ', item['text'])

print('\nTry changing exercise_query to a paraphrase of the password-reset question.')

C03 3 ['for', 'is', 'the']
  Refunds for eligible orders are returned to the original payment method after the returned item is inspected.
C05 3 ['return', 'the', 'unopened']
  Unopened products can be returned within thirty days of delivery. Include the order number in your return request.
C06 2 ['is', 'the']
  The security deposit is due when the lease is signed. The deposit amount is listed in the signed lease agreement.
C09 2 ['is', 'the']
  Read the ingredient label carefully if you have a food allergy. Contact the manufacturer when allergen information is unclear.

Try changing exercise_query to a paraphrase of the password-reset question.


### Optional extension: test error handling and ranking invariants

The following checks make explicit a few properties expected from our implementation.

In [ ]:
assert retrieve(query, corpus, k=0) == []
assert len(retrieve(query, corpus, k=100)) == len(corpus)
assert all(
    ranked[index]['score'] >= ranked[index + 1]['score']
    for index in range(len(ranked) - 1)
)
try:
    retrieve(query, corpus, k=-1)
except ValueError:
    print('Negative K is rejected as expected.')
else:
    raise AssertionError('Negative K should have raised ValueError')

## Summary

A retrieval system searches a corpus for chunks that may help answer a query. It scores and ranks candidates, then returns a top-K list. Candidate generation and ranking are distinct: a relevant item omitted from the candidate set cannot be recovered by the ranker. Chunks focus search, but their boundaries determine how much context is available. Scores are method-specific signals, not probabilities, and relevance must be judged against the information need. Precision@K and recall@K help measure retrieval on labeled examples, while failure inspection explains what to improve.

Our lexical overlap baseline is easy to understand but cannot bridge vocabulary gaps and can be fooled by misleading overlap. In the next notebook, dense retrieval uses embeddings to compare meaning beyond exact shared words. We will keep the same mental model—query, candidates, ranking, top-K, and evaluation—while replacing the scoring method.